# 04. Surrogate Model Comparison — logit(1-prob) target

03과 동일한 실험을 `logit(1-prob)` target으로 수행한다.

- 03: `transform_logit_to_score(prob)` → 정수 score (반올림 열화)
- **04: `logit(1-prob)`** → 연속 log-odds (높을수록 가점)

`1-prob`을 사용하는 이유: logit(prob)은 높을수록 부도확률 증가(감점)이지만,
`logit(1-prob)`은 높을수록 생존확률 증가(가점)로 방향이 일치.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import warnings
warnings.filterwarnings('ignore')

import pickle, time
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from scipy.stats import spearmanr

from decentra._utils import logit
from decentra.surrogate import (
    TreeSurrogate, EBMSurrogate, LinearSurrogate, OptBinningSurrogate,
)

DATA_DIR = '../.data'

## 1. 데이터 및 Base Model 로드

In [3]:
with open(f'{DATA_DIR}/base_models.pkl', 'rb') as f:
    base_models = pickle.load(f)

for name in base_models:
    s = base_models[name]['splits']
    print(f"{name:6s}  X_tr={s['X_tr'].shape}  X_val={s['X_val'].shape}  X_test={s['X_test'].shape}")

GMSC    X_tr=(96000, 10)  X_val=(24000, 10)  X_test=(30000, 10)
HC      X_tr=(196806, 41)  X_val=(49202, 41)  X_test=(61503, 41)


In [4]:
# Surrogate target: logit(1 - prob) = log((1-p)/p)
# 높을수록 가점 (생존확률 증가)
targets = {}
for name in base_models:
    bm = base_models[name]['lgb']
    s = base_models[name]['splits']
    targets[name] = {
        'y_logit_tr': logit(1 - bm.predict_proba(s['X_tr'])[:, 1]),
        'y_logit_val': logit(1 - bm.predict_proba(s['X_val'])[:, 1]),
        'y_logit_test': logit(1 - bm.predict_proba(s['X_test'])[:, 1]),
    }
    print(f"{name:6s}  logit range: [{targets[name]['y_logit_tr'].min():.2f}, {targets[name]['y_logit_tr'].max():.2f}]")

GMSC    logit range: [-1.82, 5.34]


HC      logit range: [-1.06, 5.77]


In [5]:
# BB attribution: contribution_ranking & adverse_features
# BB는 log-odds 공간: contrib > 0 → 부도확률 증가 → 감점
# Surrogate는 score 공간: contrib < 0 → 점수 감소 → 감점
# → BB adverse는 contrib > 0으로 산출해야 surrogate adverse와 대응
bb_contribs = {name: base_models[name]['bb_contribs'] for name in base_models}

bb_ranking = {}
bb_adverse = {}

for name in ['GMSC', 'HC']:
    bb_ranking[name] = {}
    bb_adverse[name] = {}
    for model_type in ['lgb', 'xgb']:
        contrib = bb_contribs[name][model_type]
        feature_names = np.array(
            base_models[name][model_type].feature_name_
            if model_type == 'lgb'
            else base_models[name][model_type].get_booster().feature_names)

        # ranking: |contrib| 내림차순 (공간 무관, 절대값 기준)
        order = np.argsort(np.abs(contrib), axis=1)[:, ::-1]
        bb_ranking[name][model_type] = feature_names[order]

        # adverse: log-odds에서 contrib > 0 = 감점 (부도확률 증가)
        # 내림차순 (가장 큰 감점부터)
        adv_rows = []
        for i in range(len(contrib)):
            pos_idx = np.where(contrib[i] > 0)[0]
            if len(pos_idx) == 0:
                adv_rows.append(np.array([], dtype='<U1'))
            else:
                pos_order = pos_idx[np.argsort(contrib[i][pos_idx])[::-1]]
                adv_rows.append(feature_names[pos_order])
        bb_adverse[name][model_type] = adv_rows

        n_has = sum(1 for r in adv_rows if len(r) > 0)
        print(f'{name} {model_type}:  ranking={bb_ranking[name][model_type].shape}  '
              f'adverse={n_has}/{len(contrib)}')

GMSC lgb:  ranking=(30000, 10)  adverse=29001/30000


GMSC xgb:  ranking=(30000, 10)  adverse=26274/30000


HC lgb:  ranking=(61503, 41)  adverse=61503/61503


HC xgb:  ranking=(61503, 41)  adverse=61500/61503


## 2. Surrogate 정의

- Tree 계열: X_tr로 학습, X_val로 early stopping
- 나머지: X_tr로 학습 (early stopping 불필요)

In [6]:
SURROGATES = {
    'Tree-d1':       TreeSurrogate(max_depth=1),
    'Tree-d3':       TreeSurrogate(max_depth=3),
    'Tree-d6':       TreeSurrogate(max_depth=6),
    'EBM-GAM':       EBMSurrogate(interactions=0),
    'EBM-GA2M':      EBMSurrogate(interactions=10),
    'Linear-Ridge':  LinearSurrogate(method='ridge'),
    'Linear-Lasso':  LinearSurrogate(method='lassocv'),
    'OptBin-Ridge':  OptBinningSurrogate(method='ridge'),
    'OptBin-Lasso':  OptBinningSurrogate(method='lassocv'),
}

## 3. 학습 및 평가

In [ ]:
def advfull_metrics(bb_adv_list, surr_adv_list):
    """AdvFull: Recall & Jaccard of adverse feature sets (feature name based).
    BB adverse: contrib > 0 in log-odds (감점).
    Surr adverse: contrib < 0 in score space (감점).
    """
    recalls, jaccards = [], []
    for i in range(len(bb_adv_list)):
        bb_set = set(bb_adv_list[i])
        su_set = set(surr_adv_list[i])
        if len(bb_set) == 0:
            continue
        inter = len(bb_set & su_set)
        recalls.append(inter / len(bb_set))
        union = len(bb_set | su_set)
        jaccards.append(inter / union if union > 0 else 0.0)
    r = round(float(np.mean(recalls)), 4) if recalls else 0.0
    j = round(float(np.mean(jaccards)), 4) if jaccards else 0.0
    return r, j


def advfull_from_contribs(contribs, bb_adv_list, feature_names):
    """AdvFull from raw contribs array (for calibrated results)."""
    surr_adv = []
    for i in range(len(contribs)):
        neg_idx = np.where(contribs[i] < 0)[0]
        if len(neg_idx) == 0:
            surr_adv.append(np.array([], dtype='<U1'))
        else:
            neg_order = neg_idx[np.argsort(contribs[i][neg_idx])]
            surr_adv.append(feature_names[neg_order])
    return advfull_metrics(bb_adv_list, surr_adv)


def evaluate_surrogate(surr, X, y, bb_ranking=None, bb_adverse=None, k=4):
    """R², Spearman, MAE, RMSE + Top-k, AdvTop-k (feature name 기반)."""
    pred = surr.predict(X)
    r2 = r2_score(y, pred)
    sp = spearmanr(y, pred)[0]
    mae = np.mean(np.abs(y - pred))
    rmse = np.sqrt(np.mean((y - pred) ** 2))
    result = {'R2': float(round(r2, 4)), 'Spearman': float(round(sp, 4)),
              'MAE': float(round(mae, 4)), 'RMSE': float(round(rmse, 4))}

    # Top-k: surr.contribution_ranking vs bb_ranking
    if bb_ranking is not None and hasattr(surr, 'contribution_ranking'):
        surr_ranking = surr.contribution_ranking(X)
        overlaps = []
        for i in range(len(X)):
            bb_set = set(bb_ranking[i, :k])
            su_set = set(surr_ranking[i, :k])
            overlaps.append(len(bb_set & su_set) / k)
        result[f'Top{k}'] = float(round(np.mean(overlaps), 4))

    # AdvTop-k: surr.adverse_features vs bb_adverse
    if bb_adverse is not None and hasattr(surr, 'adverse_features'):
        surr_adverse_list = surr.adverse_features(X)
        adv_overlaps = []
        for i in range(len(X)):
            bb_a = bb_adverse[i]
            su_a = surr_adverse_list[i]
            if len(bb_a) == 0 or len(su_a) == 0:
                continue
            ke = min(k, len(bb_a), len(su_a))
            bb_set = set(bb_a[:ke])
            su_set = set(su_a[:ke])
            adv_overlaps.append(len(bb_set & su_set) / ke)
        result[f'AdvTop{k}'] = float(round(np.mean(adv_overlaps), 4)) if adv_overlaps else 0.0

    # AdvFull: bb_adverse vs surr adverse (FIXED: was using outer-scope bb_adv)
    if bb_adverse is not None and hasattr(surr, 'adverse_features'):
        surr_adv = surr.adverse_features(X)
        recall, jaccard = advfull_metrics(bb_adverse, surr_adv)
        result['AdvFull_R'] = recall
        result['AdvFull_J'] = jaccard
    return result

### 3.1. Tree Surrogate

GMSC 실험

In [8]:
data_flag = 'GMSC'

X_tr, X_val, X_test = base_models[data_flag]['splits']['X_tr'], base_models[data_flag]['splits']['X_val'], base_models[data_flag]['splits']['X_test']
y_tr, y_val, y_test = targets[data_flag]['y_logit_tr'], targets[data_flag]['y_logit_val'], targets[data_flag]['y_logit_test']

# BB ground truth
bb_rank = bb_ranking[data_flag]['lgb']
bb_adv = bb_adverse[data_flag]['lgb']

for d in [1, 3, 6]:

    print('=' * 100)
    surr = TreeSurrogate(max_depth=d, verbose=0)
    surr.fit(X_tr, y_tr, eval_set=(X_val, y_val))
    print(f'Tree depth={d}')
    print(evaluate_surrogate(surr, X_test, y_test, bb_rank, bb_adv))

    print('-' * 100)
    surr = TreeSurrogate(max_depth=d, monotone_detect_mode='auto', verbose=0)
    surr.fit(X_tr, y_tr, eval_set=(X_val, y_val))
    print(f'Tree depth={d} + monotone')
    print(evaluate_surrogate(surr, X_test, y_test, bb_rank, bb_adv))

    if d == 1:
        print('-' * 100)
        scorecard = surr.to_scorecard_model(X_tr, feature_names=list(X_tr.columns))
        print(f'Tree depth={d} + monotone -> ScorecardModel')
        print(evaluate_surrogate(scorecard, X_test, y_test, bb_rank, bb_adv))

        print('-' * 100)
        scorecard_clean = surr.to_scorecard_model(
            X_tr, feature_names=list(X_tr.columns),
            max_bins_per_feature=10, min_bin_ratio=0.01)
        print(f'Tree depth={d} + monotone -> ScorecardModel (bin clean)')
        print(evaluate_surrogate(scorecard_clean, X_test, y_test, bb_rank, bb_adv))

    print()

Tree depth=1


{'R2': 0.9403, 'Spearman': 0.9738, 'MAE': 0.2336, 'RMSE': 0.3253, 'Top4': 0.8638, 'AdvTop4': 0.9069, 'AdvFull_R': 0.0, 'AdvFull_J': 0.0}
----------------------------------------------------------------------------------------------------


Tree depth=1 + monotone


{'R2': 0.9332, 'Spearman': 0.969, 'MAE': 0.2479, 'RMSE': 0.344, 'Top4': 0.8467, 'AdvTop4': 0.8988, 'AdvFull_R': 0.0, 'AdvFull_J': 0.0}
----------------------------------------------------------------------------------------------------


Tree depth=1 + monotone -> ScorecardModel


...

{'R2': 0.9438, 'Spearman': 0.9737, 'MAE': 0.2181, 'RMSE': 0.3155, 'Top4': 0.8663, 'AdvTop4': 0.9059, 'AdvFull_R': 0.0, 'AdvFull_J': 0.0}



Tree depth=6


{'R2': 0.9932, 'Spearman': 0.9957, 'MAE': 0.0775, 'RMSE': 0.1095, 'Top4': 0.9379, 'AdvTop4': 0.963, 'AdvFull_R': 0.0, 'AdvFull_J': 0.0}
----------------------------------------------------------------------------------------------------


Tree depth=6 + monotone


{'R2': 0.9531, 'Spearman': 0.9772, 'MAE': 0.2024, 'RMSE': 0.2883, 'Top4': 0.8759, 'AdvTop4': 0.9101, 'AdvFull_R': 0.0, 'AdvFull_J': 0.0}



HC 실험

In [9]:
data_flag = 'HC'

X_tr, X_val, X_test = base_models[data_flag]['splits']['X_tr'], base_models[data_flag]['splits']['X_val'], base_models[data_flag]['splits']['X_test']
y_tr, y_val, y_test = targets[data_flag]['y_logit_tr'], targets[data_flag]['y_logit_val'], targets[data_flag]['y_logit_test']
bb_rank = bb_ranking[data_flag]['lgb']
bb_adv = bb_adverse[data_flag]['lgb']

for d in [1, 3, 6]:
    print('=' * 100)
    surr = TreeSurrogate(max_depth=d, verbose=0)
    surr.fit(X_tr, y_tr, eval_set=(X_val, y_val))
    print(f'Tree depth={d}')
    print(evaluate_surrogate(surr, X_test, y_test, bb_rank, bb_adv))

    print('-' * 100)
    surr = TreeSurrogate(max_depth=d, monotone_detect_mode='auto', verbose=0)
    surr.fit(X_tr, y_tr, eval_set=(X_val, y_val))
    print(f'Tree depth={d} + monotone')
    print(evaluate_surrogate(surr, X_test, y_test, bb_rank, bb_adv))

    if d == 1:
        print('-' * 100)
        scorecard = surr.to_scorecard_model(X_tr, feature_names=list(X_tr.columns))
        print(f'Tree depth={d} + monotone -> ScorecardModel')
        print(evaluate_surrogate(scorecard, X_test, y_test, bb_rank, bb_adv))
        print('-' * 100)
        scorecard_clean = surr.to_scorecard_model(X_tr, feature_names=list(X_tr.columns), max_bins_per_feature=10, min_bin_ratio=0.01)
        print(f'Tree depth={d} + monotone -> ScorecardModel (bin clean)')
        print(evaluate_surrogate(scorecard_clean, X_test, y_test, bb_rank, bb_adv))
    print()


Tree depth=1


{'R2': 0.8817, 'Spearman': 0.9412, 'MAE': 0.2371, 'RMSE': 0.3065, 'Top4': 0.7757, 'AdvTop4': 0.7706, 'AdvFull_R': 0.0, 'AdvFull_J': 0.0}
----------------------------------------------------------------------------------------------------


Tree depth=1 + monotone


{'R2': 0.8484, 'Spearman': 0.9224, 'MAE': 0.2685, 'RMSE': 0.347, 'Top4': 0.5941, 'AdvTop4': 0.6027, 'AdvFull_R': 0.0, 'AdvFull_J': 0.0}
----------------------------------------------------------------------------------------------------


Tree depth=1 + monotone -> ScorecardModel


...

{'R2': 0.8664, 'Spearman': 0.9315, 'MAE': 0.2505, 'RMSE': 0.3257, 'Top4': 0.6187, 'AdvTop4': 0.6299, 'AdvFull_R': 0.0, 'AdvFull_J': 0.0}



Tree depth=6


{'R2': 0.9647, 'Spearman': 0.9814, 'MAE': 0.1283, 'RMSE': 0.1676, 'Top4': 0.8893, 'AdvTop4': 0.8726, 'AdvFull_R': 0.0, 'AdvFull_J': 0.0}
----------------------------------------------------------------------------------------------------


Tree depth=6 + monotone


{'R2': 0.8686, 'Spearman': 0.9335, 'MAE': 0.2476, 'RMSE': 0.323, 'Top4': 0.6203, 'AdvTop4': 0.6261, 'AdvFull_R': 0.0, 'AdvFull_J': 0.0}



### EBM

In [10]:
data_flag = 'GMSC'
X_tr, X_val, X_test = base_models[data_flag]['splits']['X_tr'], base_models[data_flag]['splits']['X_val'], base_models[data_flag]['splits']['X_test']
y_tr, y_val, y_test = targets[data_flag]['y_logit_tr'], targets[data_flag]['y_logit_val'], targets[data_flag]['y_logit_test']
bb_rank, bb_adv = bb_ranking[data_flag]['lgb'], bb_adverse[data_flag]['lgb']
X_train = pd.concat([X_tr, X_val], ignore_index=True)
y_train = np.concatenate([y_tr, y_val])

print('EBM interactions=0')
surr = EBMSurrogate(interactions=0)
surr.fit(X_train, y_train)
print(evaluate_surrogate(surr, X_test, y_test, bb_rank, bb_adv))
print('-' * 100)
print('EBM interactions=0 & monotone')
surr = EBMSurrogate(interactions=0, monotone_detect_mode='auto')
surr.fit(X_train, y_train)
print(evaluate_surrogate(surr, X_test, y_test, bb_rank, bb_adv))
print('-' * 100)
print('EBM interactions=0 & monotone -> ScorecardModel')
scorecard = surr.to_scorecard_model(X_train, feature_names=list(X_train.columns))
print(evaluate_surrogate(scorecard, X_test, y_test, bb_rank, bb_adv))


EBM interactions=0


{'R2': 0.9435, 'Spearman': 0.9757, 'MAE': 0.2226, 'RMSE': 0.3163, 'Top4': 0.8468, 'AdvTop4': 0.8815, 'AdvFull_R': 0.0, 'AdvFull_J': 0.0}
----------------------------------------------------------------------------------------------------
EBM interactions=0 & monotone


{'R2': 0.8844, 'Spearman': 0.9491, 'MAE': 0.3043, 'RMSE': 0.4526, 'Top4': 0.8198, 'AdvTop4': 0.8921, 'AdvFull_R': 0.0, 'AdvFull_J': 0.0}
----------------------------------------------------------------------------------------------------
EBM interactions=0 & monotone -> ScorecardModel


{'R2': 0.8801, 'Spearman': 0.9466, 'MAE': 0.3123, 'RMSE': 0.461, 'Top4': 0.8279, 'AdvTop4': 0.8886, 'AdvFull_R': 0.0, 'AdvFull_J': 0.0}


In [11]:
data_flag = 'HC'
X_tr, X_val, X_test = base_models[data_flag]['splits']['X_tr'], base_models[data_flag]['splits']['X_val'], base_models[data_flag]['splits']['X_test']
y_tr, y_val, y_test = targets[data_flag]['y_logit_tr'], targets[data_flag]['y_logit_val'], targets[data_flag]['y_logit_test']
bb_rank, bb_adv = bb_ranking[data_flag]['lgb'], bb_adverse[data_flag]['lgb']
X_train = pd.concat([X_tr, X_val], ignore_index=True)
y_train = np.concatenate([y_tr, y_val])

print('EBM interactions=0')
surr = EBMSurrogate(interactions=0)
surr.fit(X_train, y_train)
print(evaluate_surrogate(surr, X_test, y_test, bb_rank, bb_adv))
print('-' * 100)
print('EBM interactions=0 & monotone')
surr = EBMSurrogate(interactions=0, monotone_detect_mode='auto')
surr.fit(X_train, y_train)
print(evaluate_surrogate(surr, X_test, y_test, bb_rank, bb_adv))
print('-' * 100)
print('EBM interactions=0 & monotone -> ScorecardModel')
scorecard = surr.to_scorecard_model(X_train, feature_names=list(X_train.columns))
print(evaluate_surrogate(scorecard, X_test, y_test, bb_rank, bb_adv))


EBM interactions=0


{'R2': 0.911, 'Spearman': 0.9552, 'MAE': 0.2039, 'RMSE': 0.2659, 'Top4': 0.7642, 'AdvTop4': 0.786, 'AdvFull_R': 0.0, 'AdvFull_J': 0.0}
----------------------------------------------------------------------------------------------------
EBM interactions=0 & monotone


{'R2': 0.8123, 'Spearman': 0.9065, 'MAE': 0.3006, 'RMSE': 0.3861, 'Top4': 0.5365, 'AdvTop4': 0.5303, 'AdvFull_R': 0.0, 'AdvFull_J': 0.0}
----------------------------------------------------------------------------------------------------
EBM interactions=0 & monotone -> ScorecardModel


{'R2': 0.7896, 'Spearman': 0.8972, 'MAE': 0.3192, 'RMSE': 0.4088, 'Top4': 0.5394, 'AdvTop4': 0.5254, 'AdvFull_R': 0.0, 'AdvFull_J': 0.0}


### Linear Regression

In [12]:
data_flag = 'GMSC'
X_tr, X_val, X_test = base_models[data_flag]['splits']['X_tr'], base_models[data_flag]['splits']['X_val'], base_models[data_flag]['splits']['X_test']
y_tr, y_val, y_test = targets[data_flag]['y_logit_tr'], targets[data_flag]['y_logit_val'], targets[data_flag]['y_logit_test']
bb_rank, bb_adv = bb_ranking[data_flag]['lgb'], bb_adverse[data_flag]['lgb']
X_train = pd.concat([X_tr, X_val], ignore_index=True)
y_train = np.concatenate([y_tr, y_val])

for method in ['ridgecv', 'lassocv', 'ols']:
    print('=' * 100)
    print(f'Linear {method}')
    surr = LinearSurrogate(method=method)
    surr.fit(X_train, y_train)
    print(evaluate_surrogate(surr, X_test, y_test, bb_rank, bb_adv))
    print('-' * 100)
    print(f'Linear {method} & monotone')
    surr = LinearSurrogate(method=method, monotone_detect_mode='auto')
    surr.fit(X_train, y_train)
    print(evaluate_surrogate(surr, X_test, y_test, bb_rank, bb_adv))
    print()


Linear ridgecv


{'R2': 0.3193, 'Spearman': 0.5973, 'MAE': 0.8352, 'RMSE': 1.0982, 'Top4': 0.5158, 'AdvTop4': 0.4975, 'AdvFull_R': 0.0, 'AdvFull_J': 0.0}
----------------------------------------------------------------------------------------------------
Linear ridgecv & monotone


{'R2': 0.3193, 'Spearman': 0.5973, 'MAE': 0.8352, 'RMSE': 1.0982, 'Top4': 0.5158, 'AdvTop4': 0.4975, 'AdvFull_R': 0.0, 'AdvFull_J': 0.0}

Linear lassocv


{'R2': 0.3195, 'Spearman': 0.5974, 'MAE': 0.8362, 'RMSE': 1.0981, 'Top4': 0.5158, 'AdvTop4': 0.4976, 'AdvFull_R': 0.0, 'AdvFull_J': 0.0}
----------------------------------------------------------------------------------------------------
Linear lassocv & monotone


{'R2': 0.3195, 'Spearman': 0.5974, 'MAE': 0.8362, 'RMSE': 1.0981, 'Top4': 0.5158, 'AdvTop4': 0.4976, 'AdvFull_R': 0.0, 'AdvFull_J': 0.0}

Linear ols


{'R2': 0.3193, 'Spearman': 0.5973, 'MAE': 0.8351, 'RMSE': 1.0983, 'Top4': 0.5158, 'AdvTop4': 0.4975, 'AdvFull_R': 0.0, 'AdvFull_J': 0.0}
----------------------------------------------------------------------------------------------------
Linear ols & monotone


{'R2': 0.3193, 'Spearman': 0.5973, 'MAE': 0.8351, 'RMSE': 1.0983, 'Top4': 0.5158, 'AdvTop4': 0.4975, 'AdvFull_R': 0.0, 'AdvFull_J': 0.0}



In [13]:
data_flag = 'HC'
X_tr, X_val, X_test = base_models[data_flag]['splits']['X_tr'], base_models[data_flag]['splits']['X_val'], base_models[data_flag]['splits']['X_test']
y_tr, y_val, y_test = targets[data_flag]['y_logit_tr'], targets[data_flag]['y_logit_val'], targets[data_flag]['y_logit_test']
bb_rank, bb_adv = bb_ranking[data_flag]['lgb'], bb_adverse[data_flag]['lgb']
X_train = pd.concat([X_tr, X_val], ignore_index=True)
y_train = np.concatenate([y_tr, y_val])

for method in ['ridgecv', 'lassocv', 'ols']:
    print('=' * 100)
    print(f'Linear {method}')
    surr = LinearSurrogate(method=method)
    surr.fit(X_train, y_train)
    print(evaluate_surrogate(surr, X_test, y_test, bb_rank, bb_adv))
    print('-' * 100)
    print(f'Linear {method} & monotone')
    surr = LinearSurrogate(method=method, monotone_detect_mode='auto')
    surr.fit(X_train, y_train)
    print(evaluate_surrogate(surr, X_test, y_test, bb_rank, bb_adv))
    print()


Linear ridgecv


{'R2': 0.8087, 'Spearman': 0.902, 'MAE': 0.2999, 'RMSE': 0.3898, 'Top4': 0.5157, 'AdvTop4': 0.5713, 'AdvFull_R': 0.0, 'AdvFull_J': 0.0}
----------------------------------------------------------------------------------------------------
Linear ridgecv & monotone


{'R2': 0.8087, 'Spearman': 0.902, 'MAE': 0.2999, 'RMSE': 0.3898, 'Top4': 0.5157, 'AdvTop4': 0.5713, 'AdvFull_R': 0.0, 'AdvFull_J': 0.0}

Linear lassocv


{'R2': 0.8043, 'Spearman': 0.8972, 'MAE': 0.306, 'RMSE': 0.3942, 'Top4': 0.5225, 'AdvTop4': 0.5731, 'AdvFull_R': 0.0, 'AdvFull_J': 0.0}
----------------------------------------------------------------------------------------------------
Linear lassocv & monotone


{'R2': 0.8043, 'Spearman': 0.8972, 'MAE': 0.306, 'RMSE': 0.3942, 'Top4': 0.5225, 'AdvTop4': 0.5731, 'AdvFull_R': 0.0, 'AdvFull_J': 0.0}

Linear ols


{'R2': 0.8087, 'Spearman': 0.9021, 'MAE': 0.2998, 'RMSE': 0.3898, 'Top4': 0.5157, 'AdvTop4': 0.5713, 'AdvFull_R': 0.0, 'AdvFull_J': 0.0}
----------------------------------------------------------------------------------------------------
Linear ols & monotone


{'R2': 0.8087, 'Spearman': 0.9021, 'MAE': 0.2998, 'RMSE': 0.3898, 'Top4': 0.5157, 'AdvTop4': 0.5713, 'AdvFull_R': 0.0, 'AdvFull_J': 0.0}



### Optbinning

In [14]:
from decentra.surrogate import BinningSurrogate

In [15]:
data_flag = 'GMSC'
X_tr, X_val, X_test = base_models[data_flag]['splits']['X_tr'], base_models[data_flag]['splits']['X_val'], base_models[data_flag]['splits']['X_test']
y_tr, y_val, y_test = targets[data_flag]['y_logit_tr'], targets[data_flag]['y_logit_val'], targets[data_flag]['y_logit_test']
bb_rank, bb_adv = bb_ranking[data_flag]['lgb'], bb_adverse[data_flag]['lgb']
X_train = pd.concat([X_tr, X_val], ignore_index=True)
y_train = np.concatenate([y_tr, y_val])

for max_n_bins in [5, 10]:
    print('=' * 100)
    print(f'BinningSurrogate max_n_bins={max_n_bins}')
    surr = BinningSurrogate(max_n_bins=max_n_bins)
    surr.fit(X_train, y_train, feature_names=list(X_train.columns))
    print(evaluate_surrogate(surr, X_test, y_test, bb_rank, bb_adv))

print('=' * 100)
print('BinningSurrogate max_n_bins=10 & monotone')
surr = BinningSurrogate(max_n_bins=10, monotone_detect_mode='auto')
surr.fit(X_train, y_train, feature_names=list(X_train.columns))
print(evaluate_surrogate(surr, X_test, y_test, bb_rank, bb_adv))
print('-' * 100)
print('BinningSurrogate & monotone -> ScorecardModel')
scorecard = surr.to_scorecard_model(X_train, feature_names=list(X_train.columns))
print(evaluate_surrogate(scorecard, X_test, y_test, bb_rank, bb_adv))


BinningSurrogate max_n_bins=5


(CVXPY) Apr 18 11:35:12 AM: Encountered unexpected exception importing solver GLOP:
RuntimeError('Unrecognized new version of ortools (9.15.6755). Expected < 9.15.0. Please open a feature request on cvxpy to enable support for this version.')


(CVXPY) Apr 18 11:35:12 AM: Encountered unexpected exception importing solver PDLP:
RuntimeError('Unrecognized new version of ortools (9.15.6755). Expected < 9.15.0. Please open a feature request on cvxpy to enable support for this version.')


{'R2': 0.862, 'Spearman': 0.9021, 'MAE': 0.3857, 'RMSE': 0.4945, 'Top4': 0.7141, 'AdvTop4': 0.6536, 'AdvFull_R': 0.0, 'AdvFull_J': 0.0}
BinningSurrogate max_n_bins=10


{'R2': 0.9156, 'Spearman': 0.9559, 'MAE': 0.2807, 'RMSE': 0.3867, 'Top4': 0.7415, 'AdvTop4': 0.8213, 'AdvFull_R': 0.0, 'AdvFull_J': 0.0}
BinningSurrogate max_n_bins=10 & monotone


{'R2': 0.9186, 'Spearman': 0.9587, 'MAE': 0.2785, 'RMSE': 0.3799, 'Top4': 0.7948, 'AdvTop4': 0.8807, 'AdvFull_R': 0.0, 'AdvFull_J': 0.0}
----------------------------------------------------------------------------------------------------
BinningSurrogate & monotone -> ScorecardModel


{'R2': 0.9186, 'Spearman': 0.9587, 'MAE': 0.2785, 'RMSE': 0.3799, 'Top4': 0.7948, 'AdvTop4': 0.8807, 'AdvFull_R': 0.0, 'AdvFull_J': 0.0}


In [16]:
data_flag = 'HC'
X_tr, X_val, X_test = base_models[data_flag]['splits']['X_tr'], base_models[data_flag]['splits']['X_val'], base_models[data_flag]['splits']['X_test']
y_tr, y_val, y_test = targets[data_flag]['y_logit_tr'], targets[data_flag]['y_logit_val'], targets[data_flag]['y_logit_test']
bb_rank, bb_adv = bb_ranking[data_flag]['lgb'], bb_adverse[data_flag]['lgb']
X_train = pd.concat([X_tr, X_val], ignore_index=True)
y_train = np.concatenate([y_tr, y_val])

for max_n_bins in [5, 10]:
    print('=' * 100)
    print(f'BinningSurrogate max_n_bins={max_n_bins}')
    surr = BinningSurrogate(max_n_bins=max_n_bins)
    surr.fit(X_train, y_train, feature_names=list(X_train.columns))
    print(evaluate_surrogate(surr, X_test, y_test, bb_rank, bb_adv))

print('=' * 100)
print('BinningSurrogate max_n_bins=10 & monotone')
surr = BinningSurrogate(max_n_bins=10, monotone_detect_mode='auto')
surr.fit(X_train, y_train, feature_names=list(X_train.columns))
print(evaluate_surrogate(surr, X_test, y_test, bb_rank, bb_adv))
print('-' * 100)
print('BinningSurrogate & monotone -> ScorecardModel')
scorecard = surr.to_scorecard_model(X_train, feature_names=list(X_train.columns))
print(evaluate_surrogate(scorecard, X_test, y_test, bb_rank, bb_adv))


BinningSurrogate max_n_bins=5


{'R2': 0.7431, 'Spearman': 0.8486, 'MAE': 0.3601, 'RMSE': 0.4518, 'Top4': 0.4374, 'AdvTop4': 0.4647, 'AdvFull_R': 0.0, 'AdvFull_J': 0.0}
BinningSurrogate max_n_bins=10


{'R2': 0.8455, 'Spearman': 0.9204, 'MAE': 0.273, 'RMSE': 0.3503, 'Top4': 0.5594, 'AdvTop4': 0.5587, 'AdvFull_R': 0.0, 'AdvFull_J': 0.0}
BinningSurrogate max_n_bins=10 & monotone


{'R2': 0.816, 'Spearman': 0.9053, 'MAE': 0.2979, 'RMSE': 0.3823, 'Top4': 0.5156, 'AdvTop4': 0.5425, 'AdvFull_R': 0.0, 'AdvFull_J': 0.0}
----------------------------------------------------------------------------------------------------
BinningSurrogate & monotone -> ScorecardModel


{'R2': 0.816, 'Spearman': 0.9053, 'MAE': 0.2979, 'RMSE': 0.3823, 'Top4': 0.5156, 'AdvTop4': 0.5425, 'AdvFull_R': 0.0, 'AdvFull_J': 0.0}
